# M-Dimension Capacity Scaling Evaluation

This analytical review evaluates the downstream alignment impact of scaling the idiomaticity signal embedding capacity ($m \in \{0, 8, 16, 32, 64, 128, 256, 512, 1024\}$). The framework evaluates how different M-signal bottleneck widths affect held-out and cross-lingual ranking alignment under the FiLM Full query-fusion architecture.

### Methodological Boundaries
- **Architecture Isolation**: All active capacity scaling iterations ($m > 0$) maintain an identical Feature-wise Linear Modulation (**FiLM Full**) architecture to isolate the bottleneck scaling effect.
- **Reproducible Multi-Seed Cohorts**: Reported alignment distributions represent exact sample averages over the global 5-seed cohort (`7, 13, 21, 42, 100`).
- **Zero Data-Leakage Partitioning**: Evaluation matrices ensure zero literal sequence overlaps across optimization boundaries via multi-word expression grouping.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Premium tailored visual aesthetics: minimal grids, high resolution, soft tones
plt.style.use('default')
sns.set_theme(style="white", palette="muted")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

summary_path = Path("output/summary.json")

if summary_path.exists():
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    print("=== M-DIMENSION ABLATION PAYLOAD INGESTED ===")
    print(f"Aggregated Dimensions Captured: {len(summary.get('aggregate', []))}")
else:
    summary = None
    print("[Status] Target capacity scaling summary payload currently unavailable.")
    print(f"Expected Path: {summary_path}")
    print("Execute 'python run.py' to populate multi-seed evaluation records.")

In [ ]:
if summary:
    rows = []
    for agg in summary.get("aggregate", []):
        rows.append({
            "Embedding Capacity ($m$)": f"$m={agg['m_dim']}$" if agg['m_dim'] > 0 else "$m=0$ (Baseline)",
            "m_dim_val": agg['m_dim'],
            "Dev nDCG (%)": agg["best_dev"]["ndcg"] * 100.0,
            "Test nDCG (%)": agg["test"]["ndcg"] * 100.0,
            "Cross-Lingual nDCG (%)": agg["xeval"]["ndcg"] * 100.0,
            "Test Top-1 Exact (%)": agg["test"]["top1"] * 100.0,
            "Avg Optimal Epoch": agg["best_epoch"]
        })
        
    df = pd.DataFrame(rows).sort_values("m_dim_val").reset_index(drop=True)
    display_df = df.drop(columns=["m_dim_val"])
    
    print("\n=== CAPACITY SCALING PERFORMANCE MATRIX ===")
    display(display_df.round(2))

### Held-Out Alignment Parity vs Bottleneck Capacity
Compares downstream absolute testing accuracy distributions across distinct scalar projection spaces.

In [ ]:
if summary:
    fig, ax = plt.subplots(figsize=(10, 5.5))
    
    ax.grid(axis='y', linestyle='--', alpha=0.4, color='#cccccc')
    ax.set_axisbelow(True)
    
    sns.barplot(data=df, x="Embedding Capacity ($m$)", y="Test nDCG (%)", 
                palette="crest", hue="Embedding Capacity ($m$)", legend=False, ax=ax)
                
    sns.despine(left=True, bottom=False)
    ax.set_title("Held-Out Test Set Alignment Parity vs Bottleneck Resolution", fontweight="bold", fontsize=13, pad=15)
    
    y_max = df["Test nDCG (%)"].max()
    ax.set_ylim(df["Test nDCG (%)"].min() * 0.90, y_max * 1.08)
    ax.set_xlabel("")
    ax.set_ylabel("Accuracy (nDCG %)", fontweight="bold", fontsize=11)
    ax.tick_params(axis='x', labelsize=11)
    
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f"{height:.2f}%", (p.get_x() + p.get_width() / 2., height),
                         ha='center', va='bottom', xytext=(0, 6), textcoords='offset points', 
                         fontweight='bold', fontsize=10, color='#333333')
                         
    plt.tight_layout()
    plt.show()

### Continuous Capacity Scaling Interpolation Curves
Visualizes optimization trajectories across logarithmic scale resolutions to identify performance plateaus.

In [ ]:
if summary:
    fig, ax = plt.subplots(figsize=(10, 5.5))
    
    ax.grid(linestyle='--', alpha=0.4, color='#cccccc')
    ax.set_axisbelow(True)
    
    sns.lineplot(data=df, x="m_dim_val", y="Dev nDCG (%)", marker="o", linewidth=2.5, markersize=8, label="Validation nDCG", color="#2b8cbe", ax=ax)
    sns.lineplot(data=df, x="m_dim_val", y="Test nDCG (%)", marker="s", linewidth=2.5, markersize=8, label="Test nDCG", color="#31a354", ax=ax)
    
    sns.despine(left=True, bottom=False)
    ax.set_title("Dimensional Capacity Evaluation Manifold Trajectories", fontweight="bold", fontsize=13, pad=15)
    ax.set_xlabel("Idiom Signal Bottleneck Resolution ($m$)", fontweight="bold", fontsize=11)
    ax.set_ylabel("Accuracy Score (nDCG %)", fontweight="bold", fontsize=11)
    
    ax.set_xscale("symlog", base=2)
    ax.set_xticks([0, 8, 16, 32, 64, 128, 256, 512, 1024])
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.tick_params(axis='both', labelsize=11)
    
    ax.legend(frameon=True, facecolor="white", edgecolor="none", loc="lower right", fontsize=10.5)
    
    plt.tight_layout()
    plt.show()